In [ ]:
import pandas as pd # Used for Datetime Conversions
import numpy as np # Array Functions
import psycopg # Dimension Tables (Temporary)
from psycopg2.extras import execute_values # Used for Creating Fact Tables
import psycopg2 # Used for Creating Fact Tables

In [ ]:
# Establish connection to Postgres DB

conn = psycopg.connect(
    host="postgres-1.cju08ags2kn7.us-east-2.rds.amazonaws.com",
    port=5432,
    dbname="postgres",
    user="cecs_energy_lake",
    password="94_8y-g2408!gsdf?",
)

In [ ]:
# Create and Insert City Dimension Tables

create_dim_cities = """
CREATE TABLE IF NOT EXISTS dim_city (
    city_id TEXT PRIMARY KEY,
    city_name TEXT NOT NULL,
    state_code TEXT NOT NULL,
    population INTEGER,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION
);
"""

insert_cities = """
INSERT INTO dim_city (
    city_id,
    city_name,
    state_code,
    population,
    latitude,
    longitude
)
VALUES (%s, %s, %s, %s, %s, %s)
ON CONFLICT (city_id) DO NOTHING;
"""

# Execute Creation SQL
with conn.cursor() as cur:
    cur.execute(create_dim_cities)

conn.commit()

# Populate SQL Table
df = pd.read_parquet("../../local_data/gold/dim_city.parquet")
with conn.cursor() as cur:
    for row in df.itertuples(index=False):
        cur.execute(insert_cities, tuple(row))

conn.commit()
print(f"Inserted {len(df)} rows into dim_city.")

In [ ]:
# Test to Confirm Successful Implementation

with conn.cursor() as cur:
    cur.execute("SELECT * FROM dim_city LIMIT 5;")
    print(cur.fetchall())

[('chattanooga', 'chattanooga', 'Chattanooga city, Tennessee', 191496, None, None), ('clarksville', 'clarksville', 'Clarksville city, Tennessee', 185690, None, None), ('franklin', 'franklin', 'Franklin city, Tennessee', 89142, None, None), ('hendersonville', 'hendersonville', 'Hendersonville city, Tennessee', 63947, None, None), ('jackson', 'jackson', 'Jackson city, Tennessee', 69303, None, None)]


In [ ]:
create_dim_county = """
CREATE TABLE IF NOT EXISTS dim_county (
    county_id TEXT PRIMARY KEY,
    county_name TEXT NOT NULL,
    state_code TEXT NOT NULL,
    fips_code TEXT,
    population INTEGER,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION
);
"""

create_dim_grid = """
CREATE TABLE IF NOT EXISTS dim_grid (
    grid_id TEXT PRIMARY KEY,
    grid_name TEXT NOT NULL
);
"""

insert_dim_grid = """
INSERT INTO dim_grid (
    grid_id,
    grid_name
)dim_grid = dim_grid.astype(object).where(pd.notna(dim_grid), None)
dim_county = dim_county.astype(object).where(pd.notna(dim_county), None)
VALUES (%s, %s)
ON CONFLICT (grid_id) DO NOTHING;
"""

insert_dim_county = """
INSERT INTO dim_county (
    county_id,
    county_name,
    state_code,
    fips_code,
    population,
    latitude,
    longitude)

    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

dim_grid = pd.read_parquet("../../local_data/gold/dim_grid.parquet")
dim_county = pd.read_parquet("../../local_data/gold/dim_city.parquet")

dim_grid = dim_grid.astype(object).where(pd.notna(dim_grid), None)
dim_county = dim_county.astype(object).where(pd.notna(dim_county), None)

with conn.cursor() as cur:
    cur.execute(create_dim_grid)
    cur.execute(create_dim_county)

with conn.cursor() as cur:
    for row in dim_grid.itertuples(index=False):
        cur.execute(insert_dim_grid, tuple(row))

with conn.cursor() as cur:
    for row in dim_county.itertuples(index=False):
        cur.execute(insert_dim_county, tuple(row))

conn.commit()


In [ ]:
# Run SQL command to Check Length of Table
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM dim_county;")
    result = cur.fetchone()
    print(result[0])

89


In [54]:
create_dim_time_hourly = """
CREATE TABLE IF NOT EXISTS dim_time_hourly(
    time_id TIMESTAMPTZ PRIMARY KEY,
    ts_utc TIMESTAMPTZ NOT NULL,
    hour_of_day INTEGER,
    day_of_week INTEGER,
    is_weekend BOOLEAN,
    month INTEGER,
    quarter INTEGER,
    year INTEGER
);
"""

insert_dim_time_hourly = """
INSERT INTO dim_time_hourly (
    time_id,
    ts_utc,
    hour_of_day,
    day_of_week,
    is_weekend,
    month,
    quarter,
    year
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (time_id) DO NOTHING;
"""

try:
    with conn.cursor() as cur:
        cur.execute(create_dim_time_hourly)
    conn.commit()

    df_time_hourly = pd.read_parquet("../../local_data/gold/dim_time_hourly.parquet")

    df_time_hourly["time_id"] = pd.to_datetime(df_time_hourly["time_id"], utc=True, errors="coerce")
    df_time_hourly["ts_utc"] = pd.to_datetime(df_time_hourly["timestamp"], utc=True, errors="coerce")

    with conn.cursor() as cur:
        for row in df_time_hourly.itertuples(index=False):
            cur.execute(
                insert_dim_time_hourly,
                (
                    row.time_id,
                    row.ts_utc,
                    row.hour_of_day,
                    row.day_of_week,
                    row.is_weekend,
                    row.month,
                    row.quarter,
                    row.year,
                ),
            )
    conn.commit()

except Exception as e:
    conn.rollback()
    print("Original error:", e)

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM dim_time_hourly;")
    result = cur.fetchone()
    print(result[0])


43824


In [ ]:
drop_time_hourly = "DROP TABLE IF EXISTS dim_time_hourly;"

def delete_table(command:str, conn=conn):
    with conn.cursor() as cur:
        cur.execute(command)
    conn.commit()

In [ ]:
# Psycopg2 connection, as opposed to pyscopg connection.
conn = psycopg2.connect(
    host="postgres-1.cju08ags2kn7.us-east-2.rds.amazonaws.com",
    port=5432,
    dbname="postgres",
    user="cecs_energy_lake",
    password="94_8y-g2408!gsdf?",
)

In [ ]:
from psycopg2.extras import execute_values

def fetch_dataframe(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

create_fact_outage_daily = """
CREATE TABLE IF NOT EXISTS fact_outage_daily (
    time_key TIMESTAMPTZ NOT NULL,
    county_id TEXT NOT NULL,
    customers_wo_power INTEGER,
    PRIMARY KEY (time_key, county_id),
    CONSTRAINT fk_fact_outage_daily_time
        FOREIGN KEY (time_key) REFERENCES dim_time_hourly(time_id),
    CONSTRAINT fk_fact_outage_daily_county
        FOREIGN KEY (county_id) REFERENCES dim_county(county_id)
);
"""

insert_fact_outage_daily = """
INSERT INTO fact_outage_daily (
    time_key,
    county_id,
    customers_wo_power
)
VALUES %s
ON CONFLICT (time_key, county_id) DO NOTHING;
"""

df = pd.read_parquet("../../local_data/gold/fact_outage_daily.parquet")

# Map each date to midnight UTC
df["time_key"] = pd.to_datetime(df["date"], utc=True).dt.floor("D")

dim_time = fetch_dataframe(conn, """
    SELECT time_id
    FROM dim_time_hourly
""").rename(columns={"time_id": "time_key"})

dim_county = fetch_dataframe(conn, """
    SELECT county_id, county_name
    FROM dim_county
""")

# Keep only rows whose time_key exists in dim_time_hourly
df = df.merge(
    dim_time,
    on="time_key",
    how="left",
    indicator=True
)

dropped_time = df[df["_merge"] == "left_only"]
if not dropped_time.empty:
    print(f"Dropping {len(dropped_time)} rows with time_key not found in dim_time_hourly.")
    print(dropped_time[["date", "time_key"]].drop_duplicates().sort_values("time_key").head(10))

df = df[df["_merge"] == "both"].drop(columns=["_merge"])

# Resolve county_id from county name
df = df.merge(
    dim_county,
    left_on="county",
    right_on="county_name",
    how="left",
    indicator=True
)

dropped_county = df[df["_merge"] == "left_only"]
if not dropped_county.empty:
    print(f"Dropping {len(dropped_county)} rows with county not found in dim_county.")
    print(dropped_county[["county"]].drop_duplicates().sort_values("county").head(10))

df = df[df["_merge"] == "both"].drop(columns=["_merge", "county_name"])

# Final fact shape
df = df[[
    "time_key",
    "county_id",
    "customers_wo_power"
]].drop_duplicates()

rows = list(df.itertuples(index=False, name=None))

try:
    with conn.cursor() as cur:
        cur.execute(create_fact_outage_daily)
        execute_values(cur, insert_fact_outage_daily, rows, page_size=1000)

    conn.commit()
    print(f"Inserted up to {len(rows)} rows into fact_outage_daily.")
except Exception as e:
    conn.rollback()
    print("Insert failed:", e)

Dropping 24 rows with time_key not found in dim_time_hourly.
         date                  time_key
0  2020-12-31 2020-12-31 00:00:00+00:00
Inserted up to 19853 rows into fact_outage_daily.


In [ ]:
def fetch_dataframe(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

create_fact_weather_hourly = """
CREATE TABLE IF NOT EXISTS fact_weather_hourly (
    time_key TIMESTAMPTZ NOT NULL,
    city_id TEXT NOT NULL,
    temperature DOUBLE PRECISION,
    wind_speed DOUBLE PRECISION,
    precipitation DOUBLE PRECISION,
    humidity DOUBLE PRECISION,
    PRIMARY KEY (time_key, city_id),
    CONSTRAINT fk_fact_weather_hourly_time
        FOREIGN KEY (time_key) REFERENCES dim_time_hourly(time_id),
    CONSTRAINT fk_fact_weather_hourly_city
        FOREIGN KEY (city_id) REFERENCES dim_city(city_id)
);
"""

insert_fact_weather_hourly = """
INSERT INTO fact_weather_hourly (
    time_key,
    city_id,
    temperature,
    wind_speed,
    precipitation
)
VALUES %s
ON CONFLICT (time_key, city_id) DO NOTHING;
"""

df = pd.read_parquet("../../local_data/gold/fact_weather_city_hourly.parquet")
df["time_key"] = pd.to_datetime(df["time_key"], utc=True)
df["city_id"] = df["city_key"]

dim_time = fetch_dataframe(conn, "SELECT time_id FROM dim_time_hourly")
valid_time_keys = set(dim_time["time_id"])

dim_city = fetch_dataframe(conn, "SELECT city_id FROM dim_city")
valid_city_ids = set(dim_city["city_id"])

bad_time = df[~df["time_key"].isin(valid_time_keys)]
if not bad_time.empty:
    print(f"Dropping {len(bad_time)} rows with invalid time_key.")
    print(
        bad_time[["time_key"]]
        .drop_duplicates()
        .sort_values("time_key")
        .head(10)
    )

df = df[df["time_key"].isin(valid_time_keys)]

bad_city = df[~df["city_id"].isin(valid_city_ids)]
if not bad_city.empty:
    print(f"Dropping {len(bad_city)} rows with invalid city_id.")
    print(
        bad_city[["city_id"]]
        .drop_duplicates()
        .sort_values("city_id")
        .head(10)
    )

df = df[df["city_id"].isin(valid_city_ids)]

df = df[[
    "time_key",
    "city_id",
    "temperature",
    "wind_speed",
    "precipitation"
]].drop_duplicates()

rows = list(df.itertuples(index=False, name=None))

try:
    with conn.cursor() as cur:
        cur.execute(create_fact_weather_hourly)
        execute_values(cur, insert_fact_weather_hourly, rows, page_size=1000)
    conn.commit()
    print(f"Inserted up to {len(rows)} rows into fact_weather_hourly.")
except Exception as e:
    conn.rollback()
    print("Insert failed:", e)

Dropping 43824 rows with invalid city_id.
         city_id
438240  weighted
Inserted up to 438240 rows into fact_weather_hourly.


In [104]:
sql_check = """
SELECT 
    f.city_id,
    c.city_name,
    COUNT(*) AS total_records
FROM fact_weather_hourly f
JOIN dim_city c
    ON f.city_id = c.city_id
GROUP BY f.city_id, c.city_name
ORDER BY total_records DESC;
"""

with conn.cursor() as cur:
    cur.execute(sql_check)
    result = cur.fetchall()
    print(result) 

[('chattanooga', 'chattanooga', 43824), ('clarksville', 'clarksville', 43824), ('franklin', 'franklin', 43824), ('memphis', 'memphis', 43824), ('nashville', 'nashville', 43824), ('jackson', 'jackson', 43824), ('johnson_city', 'johnson_city', 43824), ('knoxville', 'knoxville', 43824), ('hendersonville', 'hendersonville', 43824), ('murfreesboro', 'murfreesboro', 43824)]


In [ ]:
create_fact_energy_load_hourly = """
CREATE TABLE IF NOT EXISTS fact_energy_load_hourly (
    time_key TIMESTAMPTZ NOT NULL,
    grid_id TEXT NOT NULL,
    load_mw DOUBLE PRECISION,
    PRIMARY KEY (time_key, grid_id),
    CONSTRAINT fk_fact_energy_load_time
        FOREIGN KEY (time_key) REFERENCES dim_time_hourly(time_id),
    CONSTRAINT fk_fact_energy_load_grid
        FOREIGN KEY (grid_id) REFERENCES dim_grid(grid_id)
);
"""

insert_fact_energy_load_hourly = """
INSERT INTO fact_energy_load_hourly (
    time_key,
    grid_id,
    load_mw
)
VALUES %s
ON CONFLICT (time_key, grid_id) DO NOTHING;
"""

# Load parquet
df = pd.read_parquet("../../local_data/gold/fact_energy_load_hourly.parquet")

# Ensure correct types
#df["time_key"] = pd.to_datetime(df["time_key"], utc=True)

# If your parquet uses grid_key instead of grid_id
#if "grid_key" in df.columns:
#    df["grid_id"] = df["grid_key"]

# Pull dimension keys
dim_time = fetch_dataframe(conn, "SELECT time_id FROM dim_time_hourly")
valid_time_keys = set(dim_time["time_id"])

dim_grid = fetch_dataframe(conn, "SELECT grid_id FROM dim_grid")
valid_grid_ids = set(dim_grid["grid_id"])

# -----------------------------
# Validate time_key
# -----------------------------
bad_time = df[~df["time_key"].isin(valid_time_keys)]
if not bad_time.empty:
    print(f"Dropping {len(bad_time)} rows with invalid time_key.")
    print(
        bad_time[["time_key"]]
        .drop_duplicates()
        .sort_values("time_key")
        .head(10)
    )

df = df[df["time_key"].isin(valid_time_keys)]

df = df[[
    "time_key",
    "source_id",
    "demand_forecast_mwh",
    "actual_demand_mwh",
    "net_gen_mwh"
]].drop_duplicates()

rows = list(df.itertuples(index=False, name=None))

try:
    with conn.cursor() as cur:
        cur.execute(create_fact_energy_load_hourly)
        execute_values(cur, insert_fact_energy_load_hourly, rows, page_size=1000)

    conn.commit()
    print(f"Inserted up to {len(rows)} rows into fact_energy_load_hourly.")
except Exception as e:
    conn.rollback()
    print("Insert failed:", e)

Dropping 43824 rows with invalid time_key.
                    time_key
0  2021-01-01 00:00:00+00:00
1  2021-01-01 01:00:00+00:00
2  2021-01-01 02:00:00+00:00
3  2021-01-01 03:00:00+00:00
4  2021-01-01 04:00:00+00:00
5  2021-01-01 05:00:00+00:00
6  2021-01-01 06:00:00+00:00
7  2021-01-01 07:00:00+00:00
8  2021-01-01 08:00:00+00:00
9  2021-01-01 09:00:00+00:00
Inserted up to 0 rows into fact_energy_load_hourly.
